In [2]:
import os
import time
import pandas as pd
from datetime import time
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pymysql
import requests
from bs4 import BeautifulSoup as bs

pymysql.install_as_MySQLdb()
load_dotenv(dotenv_path=".env_db")

def str2int(x):
    x = int(x.replace(',', ''))
    return x

def db_connect():
    engine = create_engine(f"{os.getenv('db')}+{os.getenv('dbtype')}://{os.getenv('id')}:{os.getenv('pw')}@{os.getenv('host')}/{os.getenv('database')}")
    conn = engine.connect()
    return conn

def stock_info_to_db(date, idx, code, df):
    today = str(date.today()).replace('-', '_')
    conn = db_connect()
    df.to_sql(f'{today[:7]}_stock_price_info', con=conn, if_exists='append', index=False)
    conn.close()
    
    return print(f"{today}, {inx}, {code}, {'저장완료':<30s}", end='\r')

def stock_info_scraping():
    conn = db_connect()
    data = pd.read_sql('2024_07_29_stock_company_info', con=conn)
    conn.close()

    errors = {}
    for idx, code in enumerate(zip(data['회사명'][:1], data['종목코드'][:1])):
        stock_price_detail = {}
        print(code)
        url = f"https://finance.naver.com/item/main.naver?code={code}"
        try:
            r = requests.get(url)
            print(r.status_code, f"{idx+1}/{len(data['종목코드'])} 수집중                   ", end='\r' )
            soup = bs(r.text, 'lxml')
            print(soup)
            # 가격
            price = int((soup.select_one(".today").text).strip('\n').split('\n')[1].replace(',', ""))
            # 변동금액
            price_change = int((soup.select_one(".today").text).strip('\n').split('\n')[9].replace(',', ""))
            rate_change = float(((soup.select_one(".today").text).strip('\n').split('\n')[13]+(soup.select_one(".today").text).strip('\n')[15]).replace('%', ''))
            # 변화율
            stock_price_detail.setdefault('수집일', []).append(str(date.today()))
            stock_price_detail.setdefault('회사명', []).append(company)
            stock_price_detail.setdefault('종목코드', []).append(code)                   
            stock_price_detail.setdefault('현재가', []).append(price)
            stock_price_detail.setdefault('변동금액', []).append(price_change)
            stock_price_detail.setdefault('변화율', []).append(rate_change)
            table = soup.select_one(".no_info")
            for tr in table.select("tr"):
                for td in tr.select("td"):
                    stock_price_detail.setdefault(td.select_one("span").text, []).append(str2int(td.select_one("span.blind").text))
            df = pd.DataFrame(stock_price_detail)
            stock_info_to_db(idx, code, df)
            time.sleep(5)
        except Exception as e:
            print(e)
            print(r.status_code, f"{idx+1}/{len(data['종목코드'])} 수집중", end='\r')
            errors.setdefault("에러", []).append(code) # 에러난 코드들 모음
                           

if __name__ == '__main__':
    stock_info_scraping()

('  산일전기 ', '062040')
<html lang="ko">
<head>
<title> : 네이버페이 증권</title>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="text/javascript" http-equiv="Content-Script-Type"/>
<meta content="text/css" http-equiv="Content-Style-Type"/>
<meta content="네이버페이 증권" name="apple-mobile-web-app-title"/>
<meta content="https://finance.naver.com/item/main.naver?code=" property="og:url"/>
<meta content=" - 네이버페이 증권 : 네이버페이 증권" property="og:title"/>
<meta content="관심종목의 실시간 주가를 가장 빠르게 확인하는 곳" property="og:description"/>
<meta content="https://ssl.pstatic.net/static/m/stock/im/2016/08/og_stock-200.png" property="og:image"/>
<meta content="article" property="og:type"/>
<meta content="" property="og:article:thumbnailUrl"/>
<meta content="네이버페이 증권" property="og:article:author"/>
<meta content="http://FINANCE.NAVER.COM" property="og:article:author:url"/>
<link href="https://ssl.pstatic.net/imgstock/static.pc/20240725112316/css/finance_header.css" rel="stylesheet" type="t